# 14.1 · 时间序列基础 / Time Series Basics

> **课程定位 / Where this fits**
> 第 1 课，**Part 14 · 时间序列**。一切时序分析与预测的地基。
> Lesson 1, **Part 14 · Time Series**. The foundation of all time-series analysis and forecasting.
>
> 带**时间顺序**的数据有自己的脾气：**昨天会影响今天**(自相关), 往往有**长期趋势**和**周期性季节**, 而且**绝对不能随机打乱**。要预测它, 先得读懂它的结构。本课讲清时间序列的几大核心概念——**趋势、季节性、平稳性、自相关(ACF/PACF)、差分**, 并在经典的"航空客运量"数据上**亲手观察并检验**这些性质。这些是后面所有方法(分解、ARIMA、深度学习)的共同语言。
> Data with **temporal order** has its own character: **yesterday affects today** (autocorrelation), there are often **long-term trends** and **periodic seasonality**, and you **must never shuffle it**. To forecast it, first understand its structure. This lesson covers the core concepts — **trend, seasonality, stationarity, autocorrelation (ACF/PACF), differencing** — and **observes and tests** them on the classic "airline passengers" data. These are the shared language of everything that follows.
>
> 💼 **实战/面试视角**："平稳性为什么重要 / 怎么检验(ADF) / 怎么变平稳(差分) / ACF与PACF区别 / 时序为何不能打乱" 是时序岗必考。
> 💼 **Practical/interview angle:** "why stationarity matters / test it (ADF) / make it stationary (differencing) / ACF vs PACF / why no shuffling" — time-series essentials.

> 📐 **符号约定 / Notation**
> - 趋势/季节/残差 —— 序列的三个组成部分 / trend / seasonal / residual components
> - 平稳(stationary) —— 统计性质(均值/方差)不随时间变 / statistics don't change over time
> - 差分 $\Delta y_t = y_t - y_{t-1}$ —— 相邻相减, 去趋势 / differencing removes trend

> 💡 **面试相关 / Interview-relevant**
> - "趋势/季节性/平稳性是什么"（出镜率 ★★★★★）
> - "为什么很多模型要求平稳, 怎么检验(ADF)"（★★★★★）
> - "差分如何让序列平稳"（★★★★）
> - "ACF 和 PACF 的区别与用途"（★★★★★）
> - "时间序列为什么不能随机划分训练/测试"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解时间序列的特殊性与三大成分(趋势/季节/残差)。
   Understand what makes TS special and its three components.
2. 理解**平稳性**及为何重要, 会用 **ADF 检验**。
   Understand stationarity, why it matters, use the ADF test.
3. 用**差分**把非平稳序列变平稳。
   Use differencing to make a non-stationary series stationary.
4. 读懂 **ACF/PACF**, 知道它们如何指导建模。
   Read ACF/PACF and how they guide modeling.

## 目录 / TOC
1. [时间序列的特殊性 + 看数据 ⭐](#1)
2. [平稳性与 ADF 检验 ⭐](#2)
3. [差分:让序列平稳 ⭐](#3)
4. [ACF / PACF + 小结 ⭐](#4)


<a id="1"></a>
## 1. 时间序列的特殊性 + 看数据 ⭐ / What Makes TS Special

时间序列 = 按**时间顺序**排列的一串观测值(如每月销量、每天股价)。它和普通数据有三点关键不同(面试)：
A time series = observations ordered by **time** (monthly sales, daily prices). Three key differences from ordinary data (interview):
- **有顺序、不能打乱**:样本之间**不独立**——今天的值依赖昨天。随机打乱会毁掉时间结构(这也是为什么训练/测试**必须按时间切**, 不能随机划分)。
  **Ordered, no shuffling:** samples are **not independent** — today depends on yesterday. Shuffling destroys the temporal structure (hence train/test must be **split chronologically**).
- **常含趋势和季节**:**趋势(trend)** 是长期上升/下降;**季节性(seasonality)** 是固定周期的重复(每年夏天客流高峰)。
  **Often has trend & seasonality:** **trend** = long-term up/down; **seasonality** = fixed-period repetition (summer peaks each year).
- **自相关(autocorrelation)**:序列与"自己过去的值"相关——这正是能预测未来的基础。
  **Autocorrelation:** the series correlates with **its own past** — the very basis for forecasting.

经典数据 **AirPassengers**(1949–1960 月度航空客运量, 时序教学的"hello world")。
Classic data **AirPassengers** (monthly airline passengers 1949–1960, the "hello world" of TS).


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# 内嵌经典 AirPassengers 数据(1949-01 ~ 1960-12 月度客运量, 千人) / embedded AirPassengers monthly totals
ap = [112,118,132,129,121,135,148,148,136,119,104,118, 115,126,141,135,125,149,170,170,158,133,114,140,
      145,150,178,163,172,178,199,199,184,162,146,166, 171,180,193,181,183,218,230,242,209,191,172,194,
      196,196,236,235,229,243,264,272,237,211,180,201, 204,188,235,227,234,264,302,293,259,229,203,229,
      242,233,267,269,270,315,364,347,312,274,237,278, 284,277,317,313,318,374,413,405,355,306,271,306,
      315,301,356,348,355,422,465,467,404,347,305,336, 340,318,362,348,363,435,491,505,404,359,310,337,
      360,342,406,396,420,472,548,559,463,407,362,405, 417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range("1949-01", periods=len(ap), freq="MS")        # 月度时间索引 / monthly index
ts = pd.Series(ap, index=idx, name="passengers")
print(f"AirPassengers: {len(ts)} 个月度观测 (1949-01 ~ 1960-12)")

fig, ax = plt.subplots(figsize=(11, 4)); ts.plot(ax=ax)
ax.set_xlabel("年份"); ax.set_ylabel("客运量(千人)"); ax.set_title("航空客运量: 明显的上升趋势 + 每年重复的季节性波峰")
plt.tight_layout(); plt.show()
print("肉眼可见: ①长期上升(趋势) ②每年夏天的规律波峰(季节性) ③波动幅度随水平增大(乘法型季节)")
print("时序铁律: 样本不独立(今天依赖昨天), 不能随机打乱; 训练/测试必须按时间切(过去训, 未来测)")


<a id="2"></a>
## 2. 平稳性与 ADF 检验 ⭐ / Stationarity & the ADF Test

**平稳性(stationarity)** 是时序里最重要的概念之一(面试核心)。一个序列**平稳**, 大致指它的**统计性质(均值、方差、自相关)不随时间改变**——没有趋势、没有变化的波动幅度, 看起来"哪一段都差不多"。
**Stationarity** is one of the most important TS concepts (interview core). A series is **stationary** if its **statistical properties (mean, variance, autocorrelation) don't change over time** — no trend, no changing variance, "every segment looks similar."

**为什么重要**:很多经典模型(尤其 ARIMA)**假设序列平稳**才能工作——因为它们靠"过去的规律在未来也成立"来预测;如果均值一直在涨(非平稳), 模型学到的"规律"就会失效。所以建模前要**先检验、再想办法变平稳**。
**Why it matters:** many classical models (especially ARIMA) **assume stationarity** — they forecast by "past patterns hold in the future"; if the mean keeps rising (non-stationary), learned patterns break. So we **test first, then make it stationary**.

**怎么检验**:① **看图**——滚动均值/方差是否随时间变;② **ADF 检验(Augmented Dickey-Fuller)**——一个假设检验, **原假设 = "序列非平稳"**。若 **p 值 < 0.05**, 拒绝原假设 → **平稳**。
**How to test:** ① **plot** rolling mean/variance over time; ② the **ADF test (Augmented Dickey-Fuller)** — a hypothesis test whose **null = "non-stationary"**. If **p-value < 0.05**, reject the null → **stationary**.


In [ ]:
from statsmodels.tsa.stattools import adfuller

def check_stationarity(series, name):
    roll_mean = series.rolling(12).mean(); roll_std = series.rolling(12).std()   # 12个月滚动统计 / rolling stats
    p = adfuller(series.dropna())[1]                                             # ADF 检验的 p 值 / ADF p-value
    return roll_mean, roll_std, p

fig, ax = plt.subplots(figsize=(11, 4))
ts.plot(ax=ax, label="原始序列", alpha=0.6)
rm, rs, p = check_stationarity(ts, "原始")
rm.plot(ax=ax, label="滚动均值(12月)", color="red"); rs.plot(ax=ax, label="滚动标准差", color="green")
ax.legend(); ax.set_title(f"原始序列: 滚动均值持续上升 → 非平稳 (ADF p值={p:.3f} > 0.05, 不能拒绝'非平稳')")
plt.tight_layout(); plt.show()
print(f"原始序列 ADF p值 = {p:.3f}  (>0.05 → 非平稳: 有趋势, 均值随时间上升)")
print("滚动均值一路上扬、滚动标准差也在变 → 统计性质随时间变 = 非平稳; 需要处理(下面差分)")


<a id="3"></a>
## 3. 差分:让序列平稳 ⭐ / Differencing: Make It Stationary

**差分(differencing)** 是把非平稳序列变平稳最常用的手段: **相邻两项相减** $\Delta y_t = y_t - y_{t-1}$。直觉: 关注"**变化量**"而非"绝对值"——绝对值一直涨(非平稳), 但"每月变化多少"往往围绕一个稳定的水平波动(平稳)。这就是 ARIMA 里的 **"I"(Integrated, 差分)**。
**Differencing** is the most common way to make a series stationary: subtract adjacent terms $\Delta y_t = y_t - y_{t-1}$. Intuition: focus on the **change** rather than the level — the level keeps rising (non-stationary), but "how much it changes each month" often fluctuates around a stable level (stationary). This is the **"I" (Integrated) in ARIMA**.

- **一阶差分**去掉**线性趋势**。
  **First differencing** removes a **linear trend**.
- **季节差分**($y_t - y_{t-12}$, 隔一个季节周期相减)去掉**季节性**。
  **Seasonal differencing** ($y_t - y_{t-12}$) removes **seasonality**.
- 对**乘法型**(波动随水平增大)的序列, 常先取 **对数**再差分, 让波动幅度稳定。
  For **multiplicative** series (variance grows with level), often take the **log** first, then difference, to stabilize variance.


In [ ]:
# 取对数稳定方差, 再做差分去趋势/季节 / log to stabilize variance, then difference
ts_log = np.log(ts)
diff1 = ts_log.diff()                          # 一阶差分: 去趋势 / first difference (remove trend)
diff_seasonal = diff1.diff(12)                 # 再季节差分(lag 12): 去年度季节 / + seasonal difference

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, s, t in [(axes[0], ts_log, "log(序列): 趋势仍在(非平稳)"),
                 (axes[1], diff1, "+ 一阶差分: 趋势没了, 但季节波动仍明显"),
                 (axes[2], diff_seasonal, "+ 季节差分(lag12): 接近平稳的随机波动")]:
    s.plot(ax=ax); ax.set_title(t)
plt.tight_layout(); plt.show()
for name, s in [("log原始", ts_log), ("一阶差分", diff1), ("一阶+季节差分", diff_seasonal)]:
    p = adfuller(s.dropna())[1]
    print(f"  {name:14}: ADF p值 = {p:.4f}  {'→ 平稳 ✓' if p<0.05 else '→ 非平稳'}")
print("\n差分把'绝对值'变成'变化量' → 去掉趋势/季节后, 序列变平稳(ADF p<0.05) → 才好建模")


<a id="4"></a>
## 4. ACF / PACF + 小结 ⭐ / ACF & PACF

**自相关函数 ACF(autocorrelation)**:序列与**自己滞后 $k$ 期**的相关性。ACF 在某些滞后处很高, 说明"$k$ 期前的值对现在有影响"——这是预测的依据, 也帮我们发现季节周期(如 lag 12 处的尖峰 = 年度季节)。
**ACF (autocorrelation function):** correlation of the series with **itself lagged by $k$**. High ACF at some lag means "the value $k$ steps ago influences now" — the basis for forecasting and for spotting seasonality (a spike at lag 12 = yearly season).

**偏自相关函数 PACF(partial autocorrelation)**:滞后 $k$ 期的相关性, 但**剔除了中间各期的间接影响**——只保留"$k$ 期前对现在的**直接**影响"。
**PACF (partial autocorrelation):** correlation at lag $k$ but **removing the indirect effects of intermediate lags** — only the **direct** influence of lag $k$.

**为什么要两个(面试核心)**:它们用来给 ARIMA **定阶**——
**Why both (interview core):** they help choose ARIMA orders —
- **PACF** 在 $p$ 阶后截断(突然掉到 0) → 提示 **AR(p)** 的阶数 $p$。
  **PACF** cutting off after lag $p$ → suggests the **AR(p)** order.
- **ACF** 在 $q$ 阶后截断 → 提示 **MA(q)** 的阶数 $q$。
  **ACF** cutting off after lag $q$ → suggests the **MA(q)** order.


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
plot_acf(ts, lags=36, ax=axes[0,0]); axes[0,0].set_title("ACF(原始): 缓慢衰减+lag12周期性 → 有趋势和季节")
plot_pacf(ts, lags=36, ax=axes[0,1], method="ywm"); axes[0,1].set_title("PACF(原始)")
plot_acf(diff_seasonal.dropna(), lags=36, ax=axes[1,0]); axes[1,0].set_title("ACF(差分后): 大部分落入置信带 → 平稳")
plot_pacf(diff_seasonal.dropna(), lags=36, ax=axes[1,1], method="ywm"); axes[1,1].set_title("PACF(差分后)")
plt.tight_layout(); plt.show()
print("原始ACF缓慢衰减(趋势)+在lag12附近有峰(年度季节); 差分后ACF快速落入置信区间→平稳")
print("ACF/PACF 用途: 给ARIMA定阶 — PACF p阶后截断→AR(p); ACF q阶后截断→MA(q) (14.4 详用)")


```
时间序列特殊性: 有顺序不能打乱(样本不独立) + 常有趋势/季节 + 自相关(可预测的基础)
三成分: 趋势(长期升降) + 季节性(固定周期重复) + 残差(随机)
平稳性: 统计性质(均值/方差/自相关)不随时间变; 很多模型(ARIMA)要求平稳才能用
检验平稳: 看滚动均值/方差 + ADF检验(原假设=非平稳, p<0.05则平稳)
差分: Δy_t=y_t-y_{t-1} 去趋势; 季节差分 y_t-y_{t-12} 去季节; 乘法型先取log; =ARIMA的'I'
ACF: 与自己滞后k期的相关(含间接); PACF: 剔除中间影响的直接相关; 用来给ARIMA定阶
铁律: 训练/测试按时间切(过去训未来测), 绝不随机打乱(防未来信息泄漏)
```

### 💡 面试速查 / Interview cheat-sheet
1. **三成分**: 趋势/季节性/残差; 时序=它们的叠加(加法或乘法)。
   Three components: trend/seasonality/residual; series = their (additive/multiplicative) combination.
2. **平稳性**: 统计性质不随时间变; ARIMA等要求平稳; ADF检验(p<0.05平稳)。
   Stationarity: time-invariant statistics; required by ARIMA; ADF test (p<0.05 stationary).
3. **差分**: 相邻相减去趋势; 季节差分去季节; ARIMA的I。
   Differencing: subtract adjacent to remove trend; seasonal diff for seasonality; the I in ARIMA.
4. **ACF vs PACF**: ACF含间接影响, PACF只直接影响; 给ARIMA定阶。
   ACF vs PACF: ACF includes indirect, PACF only direct; for ARIMA order selection.
5. **不能打乱**: 样本不独立, 训练/测试按时间切, 防未来泄漏。
   No shuffling: samples dependent; chronological split; prevent future leakage.

### 下一节 / Next
**14.2 时间序列分解**——把一个序列**拆成趋势 + 季节 + 残差**三部分, 看清各自的形状。我们会讲加法vs乘法分解, 并用 **STL**(更稳健的现代分解方法)分解航空客运量。
**14.2 Decomposition** — split a series into **trend + seasonality + residual** to see each clearly. We'll cover additive vs multiplicative decomposition and use **STL** (a robust modern method) on the airline data.
